In [4]:
#Библиотеки
from math import ceil, sqrt
from pathlib import Path # Библиотека для работы с файлами и их адресами
from scipy.stats import t
from scipy.stats import norm
from scipy.stats import chi2
import math # Библиотка для математический действий
import matplotlib.pyplot as plt # библиотека для рисования графико и диаграмм в в питоне.
import pandas as pd # Красивые таблицы в питоне
from random import randint
pd.set_option('display.max_columns', None)   # показать все столбцы
pd.set_option('display.width', 200)          # ширина вывода
pd.set_option('display.max_colwidth', 30)    # ширина столбцов

def make_intervals (numbers, k=7): # построение интервального ряда из лабы 1.1: k групп с равными интервалами
    x_min = min(numbers)
    x_max = max(numbers)
    h = math.ceil((x_max - x_min) / k) # длина одного интервала
    edges = [x_min + i * h for i in range(k + 1)] # границы интервалов
    return [[edges[i], edges[i + 1]] for i in range(k)]

def make_intervals_unit(data):# построение интервального ряда средних: от floor(min) до ceil(max), длина интервала 1 год
    left = math.floor(min(data))
    right = math.ceil(max(data))
    intervals = []
    for x in range(left, right):
        intervals.append([x, x + 1])
    return intervals

def calculate_n (d):# Высчитываем объем группы для одной выборки
   y = 0.95 # надежность из условия из условия
   b = 3 # точность оценки математического ожидания из условия
   f_t = (y+1) / 2
   t = norm.ppf(f_t) 
   n = t**2 * d**2 / b**2
   n = int(math.ceil(n))  # округляем в большую сторону чтобы n было целым числом
   return n

# Создание групп и нахождение средних
def lets_make_a_groops (numbers, n, N): # создаем 36 групп по полученному n (объему выборки) из функции calculate_n
    k = 36 # количество групп из условия
    all_selections = [] # список со списками выборок, 36 штук
    mid = [] # среднии для каждой выборки
    for selection in range(k):
        one = [] # одна выборка
        sum_ = 0 # сумма для подсчета среднего
        for num in range(n):
            i = randint(0, N - 1) # выбор случайных чисел по условию репрезитативности выборки
            one.append(numbers[i]) # добавление элемента выборку
            sum_ += numbers[i]
        mid.append(sum_ / len(one)) # подсчет средней
        all_selections.append(one) # добавление одной выборки в список выборок
    return all_selections, mid # возвращает картеж со списком списокв выборок и списком средних

# Общая функция для высчитывания частот для обоих рядов
def caclulating_chastots(numbers, all_someth, mids=None): 
    chastots = []
    data = mids if mids is not None else numbers # услоиве на другое поведение если в функцию передаются средние
    for i, interval in enumerate(all_someth):
        k = 0
        if i == len(all_someth) - 1:
            for val in data:
                if interval[0] <= val <= interval[1]:# включаем последний элемент
                    k += 1
            chastots.append(k)
        else:
            for val in data:
                if interval[0] <= val < interval[1]:
                    k += 1
            chastots.append(k)
    return chastots

# Общая функция для высчитывания СКО и среднего  для обоих рядов
def calulate_SKO (chastots, intervals, N):
    mids = []
    for interval in intervals:
        mids.append((interval[0] + interval[-1])/2)
    midle_x = sum([ mids[i] * chastots[i]   for i in range(0, len(mids))]) / N # среднее
    sko = sqrt(sum( [chastots[i] * (mids[i] - midle_x)**2   for i in range(0, len(mids))]) / N) # СКО
    return midle_x, sko

# Общая функция нормирования для обоих рядов
def normirovanie(x_midle, d, intervals):
    # Начинаем с левой границы первого интервала
    normirovan_raw = [(intervals[0][0] - x_midle) / d]
    # Добавляем правую границу каждого интервала
    for interval in intervals:
        normirovan_raw.append((interval[1] - x_midle) / d)
    # Крайние значения — бесконечности
    normirovan_raw[0] = -math.inf
    normirovan_raw[-1] = math.inf
    return normirovan_raw

# Общая функция для лапласа
def laplace(z_val):
    return norm.cdf(z_val) - 0.5

# Общая функция для вычисления вероятностей для нормированного ряда
def laplace_function (z):
    val_laplse = [laplace(z[i]) for i in range(len(z) - 1)]
    p = [laplace(z[i+1]) - laplace(z[i]) for i in range(len(z) - 1)]
    return p, val_laplse
# Общая функция для вычисления теоретического ряда
def teor_chastots_calc (p, n):
    teor = []
    for i in range(0, len(p)):
        teor.append(p[i] * n)
    return teor
# Общая функция для вывода ответа 
def considering_x (x_nabl, x_kr):
    answer = ""
    if(x_nabl < x_kr):
        answer = f"X(наблюдаемое)=={x_nabl:.4f} < X(критического)=={x_kr:.4f}, значит нет оснований отвергнуть нулевую гипотезу (данные согласуются с нормальным распределением)."
    else:
        answer = f"X(наблюдаемое)=={x_nabl:.4f} >= X(критического)=={x_kr:.4f}, значит нулевая гипотеза отвергается (данные не согласуются с нормальным распределением).."
    return answer

# Общая функция для вычисления критериев 
def calc_values_of_criteries(chastots, teor, alpha):
    x_nabl = sum([(chastots[i] - teor[i])**2 / teor[i] for i in range(len(chastots))])
    k = len(chastots) - 3 # k = s - 1 - r, r = 2 (оцениваем среднее и СКО)
    x_kr = chi2.ppf(1 - alpha, k)
    return x_nabl, x_kr, k, considering_x(x_nabl, x_kr)

# Функция для вывода таблицы с ответами
def print_table(intervals, norm_raw, laplace_value, p, imper_chastots, teor_chastots, answer):
    data = []
    for i in range(len(intervals)):
        if i == len(intervals) - 1:
            string = [
            f"[{intervals[i][0]:.2f}, {intervals[i][1]:.2f}]",
            f"({norm_raw[i]:.2f}, {norm_raw[i+1]:.2f})",
            f"{laplace_value[i]:.4f}",
            f"{p[i]:.4f}",
            imper_chastots[i],
            f"{teor_chastots[i]:.2f}",]
        else:
            string = [
            f"[{intervals[i][0]:.2f}, {intervals[i][1]:.2f})",
            f"({norm_raw[i]:.2f}, {norm_raw[i+1]:.2f})",
            f"{laplace_value[i]:.4f}",
            f"{p[i]:.4f}",
            imper_chastots[i],
            f"{teor_chastots[i]:.2f}",]
        data.append(string)
    df = pd.DataFrame(data, columns=[
        "Интервалы", "Норм. границы", "Ф(z)", "Вероятности",
        "Эмпир. частоты", "Теор. частоты"
    ])
    print(df)
    print("-" * 64)
    print("Проверка нулевой гипотезы:")
    print(answer)
    

# Проверка нулевой гипотезы о нормальном распределении по критерию Пирсона
# intervals — интервальный ряд, chastots — эмпирические частоты, alpha — уровень значимости
def pearson_test(intervals, chastots, alpha, title):
    n = sum(chastots)
    midx, sko = calulate_SKO(chastots, intervals, n) # среднее и СКО
    norm_raw = normirovanie(midx, sko, intervals) # нормированные границы
    p, laplace_val = laplace_function(norm_raw) # вероятности и значения Лапласа
    teor = teor_chastots_calc(p, n) # теоретические частоты
    if len(chastots) - 3 < 1: # при s < 4 степеней свободы нет, критерий не применим
        print(title)
        print(f"Интервалов s = {len(chastots)}, k = s - 3 < 1: критерий Пирсона неприменим")
        return None
    x_nabl, x_kr, k, answer = calc_values_of_criteries(chastots, teor, alpha)
    print(title)
    print(f"n = {n}, x_ср = {midx:.4f}, СКО = {sko:.4f}, α = {alpha}, число интервалов s = {len(intervals)}, k = {k}")
    print_table(intervals, norm_raw, laplace_val, p, chastots, teor, answer)
    return x_nabl, x_kr

def inteval_raw_building(numbers, n, N, alpha=0.05):
    # === Ряд № 1: возраст ===
    intervals_1 = make_intervals(numbers) # получение интервального ряда
    ch_n = caclulating_chastots(numbers, intervals_1)# вычисление частот
    pearson_test(intervals_1, ch_n, alpha, "-----Ряд № 1: Возраст-----")

    # === Ряд № 2: средний возраст ===
    all_selections, mid = lets_make_a_groops(numbers, n, N)
    intervals_2 = make_intervals_unit(mid) # интервалы длиной 1 год
    ch_mids = caclulating_chastots(mid, intervals_2) # вычисление частот
    pearson_test(intervals_2, ch_mids, alpha, "----Ряд № 2: Средний возраст-----")

fp = Path("./Москва_2021.txt") 
if fp.exists():
    with open(fp, "r", encoding="utf-8") as f: # Открыть и прочитать файл, на любом языке с переменной файла f
        numbers = sorted([int(line.strip()) for line in f if line.strip()])
    sn = set(numbers)
    N = 0 # Сумма частот
    x_midl = 0 # Среднее
    for i in sn:
        n = numbers.count(i) # кол. элемента
        x_midl += i * n # высчитываем сумму среднего, но пока без деления на сумму частот
        N+= n
    x_midl /= N
    D = 0
    for i in sn:
        D += (i - x_midl)**2 * numbers.count(i)
    D = D / N # Дисперсия
    d = math.sqrt(D) # среднее квадратичное отклонение
    n = calculate_n(d) # высчитываем объем
    inteval_raw_building(numbers, n, N) # получаем список списков групп + список средних
    
else:
    print("Скачайте или создайте файл Москва_2021.txt")

-----Ряд № 1: Возраст-----
n = 32423, x_ср = 35.6309, СКО = 12.1875, α = 0.05, число интервалов s = 7, k = 4
        Интервалы   Норм. границы     Ф(z) Вероятности  Эмпир. частоты Теор. частоты
0  [14.00, 23.00)   (-inf, -1.04)  -0.5000      0.1500            4811       4863.87
1  [23.00, 32.00)  (-1.04, -0.30)  -0.3500      0.2329            9476       7550.33
2  [32.00, 41.00)   (-0.30, 0.44)  -0.1171      0.2873            7243       9316.57
3  [41.00, 50.00)    (0.44, 1.18)   0.1702      0.2106            7951       6827.43
4  [50.00, 59.00)    (1.18, 1.92)   0.3808      0.0916            1140       2970.25
5  [59.00, 68.00)    (1.92, 2.66)   0.4724      0.0236            1472        766.32
6  [68.00, 77.00]     (2.66, inf)   0.4960      0.0040             330        128.22
----------------------------------------------------------------
Проверка нулевой гипотезы:
X(наблюдаемое)==3233.2657 >= X(критического)==9.4877, значит нулевая гипотеза отвергается (данные не согласуются с норм

In [ ]:
1

1